# MasterAI: öz modelini vLLM ilə serve et (gün 17)

Gün 16-da öyrədilmiş modeli (`masterai-mcq-merged`) **vLLM** ilə OpenAI-uyğun API kimi açırıq
və sürətini ölçürük. vLLM-in iki əsas fikri:

- **PagedAttention:** KV-keş sabit səhifələrlə saxlanır (əməliyyat sistemindəki virtual yaddaş kimi),
  boş qalan yaddaş azalır, eyni GPU-ya daha çox paralel sorğu sığır.
- **Continuous batching:** yeni sorğu cari batch-ın bitməsini gözləmir, növbəti addımda qoşulur.

Plan: (1) HF `generate` ilə baza sürət → (2) vLLM serveri → (3) eyni promptlarla 1/4/8 paralel ölçmə →
(4) Cloudflare tuneli ilə MasterAI serverinə qoşmaq. Runtime: **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install "vllm==0.6.6.post1"
!git clone -q https://github.com/Alihuseyn-ibrahimov/masterai.git
!pip -q install pydantic langchain-core langchain-text-splitters langgraph langsmith python-dotenv httpx

## 1. Model
Gün 16 notebook-u modeli `masterai-mcq-merged/`-ə yazır: Google Drive-a köçür və ya Hugging Face-ə yüklə. Fine-tune hələ yoxdursa, baza modellə də işləyir (müqayisə üçün).

In [ ]:
import os, secrets

MODEL = "masterai-mcq-merged"            # və ya Hugging Face: "<hf-istifadəçi>/masterai-mcq-3b"
if MODEL == "masterai-mcq-merged" and not os.path.exists(MODEL):
    from google.colab import drive
    drive.mount("/content/drive")
    MODEL = "/content/drive/MyDrive/masterai-mcq-merged"
    if not os.path.exists(MODEL):
        print("Fine-tune tapılmadı, baza model işlədilir")
        MODEL = "Qwen/Qwen2.5-3B-Instruct"
API_KEY = secrets.token_urlsafe(24)
print(MODEL)

## 2. Baza ölçü: HF transformers `generate` (vLLM-dən ƏVVƏL, GPU yaddaşı boş olanda)
Bir-bir 3 prompt: saniyədə neçə token.

In [ ]:
import sys, time, torch
sys.path.insert(0, "masterai")
from transformers import AutoModelForCausalLM, AutoTokenizer
from ml.bench_llm import sample_prompts

prompts = sample_prompts(None, 1) * 3
tok = AutoTokenizer.from_pretrained(MODEL)
hf = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")
tokens, started = 0, time.perf_counter()
for system, user in prompts:
    text = tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to("cuda")
    out = hf.generate(**ids, max_new_tokens=800, do_sample=False)
    tokens += out.shape[1] - ids["input_ids"].shape[1]
hf_tps = tokens / (time.perf_counter() - started)
print(f"HF generate: {hf_tps:.1f} token/san (ardıcıl)")
del hf; torch.cuda.empty_cache()

## 3. vLLM serveri
T4 bf16 dəstəkləmir: `--dtype half`. `--max-model-len 4096` 6000 simvolluq material parçasına + cavaba bəs edir.

In [ ]:
import subprocess, time, urllib.request

log = open("vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", MODEL, "--served-model-name", "masterai-mcq", "--dtype", "half",
    "--max-model-len", "4096", "--gpu-memory-utilization", "0.85", "--api-key", API_KEY, "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("vLLM hazırdır"); break
    except Exception:
        time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

In [ ]:
import httpx
r = httpx.post("http://localhost:8000/v1/chat/completions", headers={"Authorization": f"Bearer {API_KEY}"},
               json={"model": "masterai-mcq", "max_tokens": 300,
                     "messages": [{"role": "user", "content": "Konstitusiya haqqında bir test sualı yaz."}]},
               timeout=120)
print(r.json()["choices"][0]["message"]["content"])

## 4. Sürət: 1 / 4 / 8 paralel sorğu
Eyni promptlar `ml/bench_llm.py` ilə. Gözlənilən: tək sorğu HF-dən bir qədər sürətli, paralel yükdə **cəmi** ötürmə qabiliyyəti bir neçə dəfə artır.

In [ ]:
!cd masterai && python -m ml.bench_llm --url http://localhost:8000 --model masterai-mcq --api-key {API_KEY} \
    --concurrency 1,4,8 --requests 16 --out ../vllm_bench.json
import json
best = max(level["throughput_tokens_per_s"] or 0 for level in json.load(open("vllm_bench.json"))["levels"])
print(f"HF generate: {hf_tps:.1f} tok/s  ·  vLLM ən yaxşı cəmi: {best:.1f} tok/s  ·  {best / hf_tps:.1f}x")

## 5. MasterAI serverinə qoş (Cloudflare tuneli)
Hesabsız `trycloudflare.com` ünvanı. Colab sessiyası bağlananda ünvan ölür: `.env`-də `LLM_ORDER=vllm,groq,claude` olduğu üçün server sükutla Groq-a keçir.

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re, subprocess, time
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0); break
print("Serverdə /opt/masterai/.env-ə yaz və `systemctl restart masterai`:\n")
print(f"VLLM_BASE_URL={url}\nVLLM_API_KEY={API_KEY}\nVLLM_MODEL=masterai-mcq\nLLM_ORDER=vllm,groq,claude")

## 6. Serverdə yoxla

```bash
cd /opt/masterai
.venv/bin/python -m ml.bench_llm --concurrency 1,4      # .env-dəki VLLM_* ilə, tunel üzərindən
.venv/bin/python -m ml.bench_llm --groq --concurrency 1,4   # müqayisə
LLM_ORDER=vllm .venv/bin/python -m evals.run --live      # keyfiyyət: yalnız öz modelimizlə
```
API açarı tuneldə vacibdir: ünvanı bilən hər kəs GPU-nu işlədə bilər.